# 🎨 Qwen-Image-2.1 Uncensored (GGUF) 초간단 원클릭 생성기

복잡한 노드 설정 없이 **프롬프트만 입력하면 AI가 이미지를 바로 생성해주는 심플 웹 인터페이스(WebUI)**입니다.

---
### 📌 특징
- **노드 연결 불필요**: 내부 엔진이 백그라운드에서 자동 처리되며, 사용자에게는 깔끔한 입력창만 제공됩니다.
- **Colab 화면 내 즉시 실행**: Colab 노트북 안에서도 바로 이미지를 만들 수 있고, 별도 공유 링크(`gradio.live`)로도 접속 가능합니다.
- **T4 GPU 최적화**: Q4_0 양자화 모델과 INT8 텍스트 인코더로 무료 T4 GPU에서 메모리 부족(OOM) 없이 안정적 구동.

👉 **상단 메뉴에서 `런타임 > 모두 실행(Run all)`**을 클릭하시면 모든 과정이 전자동으로 진행됩니다.

### 1단계: GPU 환경 확인
Colab 상단 메뉴 `런타임 > 런타임 유형 변경`에서 **T4 GPU**로 설정되어 있는지 확인합니다.

In [ ]:
!nvidia-smi

### 2단계: 필수 패키지 및 엔진 설치
- 초고속 다운로더(`aria2`) 및 웹 인터페이스(`gradio`) 설치
- 백엔드 엔진(ComfyUI + ComfyUI-GGUF) 자동 세팅

In [ ]:
# 1. 필수 라이브러리 및 Gradio 설치
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q gradio Pillow gguf huggingface_hub

# 2. ComfyUI 백엔드 클론 및 설치
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || true
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# 3. Qwen-Image-2.1 네이티브 지원 GGUF 노드 설치
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/leejet/ComfyUI-GGUF.git 2>/dev/null || true
%cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q -r requirements.txt
%cd /content/ComfyUI
print('✅ 환경 설치 완료!')

### 3단계: Qwen-Image-2.1 Uncensored 모델 파일 다운로드
이미 다운로드되어 있다면 건너뛰며, 없으면 고속으로 다운로드합니다.

In [ ]:
import os

os.makedirs('/content/ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('/content/ComfyUI/models/unet', exist_ok=True)
os.makedirs('/content/ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('/content/ComfyUI/models/vae', exist_ok=True)

ua = '--user-agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"'

# 1. Diffusion Model (Q4_0 GGUF)
diff_path = '/content/ComfyUI/models/diffusion_models/qwen-image-2.1-UC-Q4_0.gguf'
if not os.path.exists(diff_path) or os.path.getsize(diff_path) < 1000000:
    print('📥 [1/3] Diffusion Model: qwen-image-2.1-UC-Q4_0.gguf 다운로드 중...')
    !aria2c --console-log-level=error -c -x 8 -s 8 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/qwen-image-2.1-UC-Q4_0.gguf' \
      -d /content/ComfyUI/models/diffusion_models -o qwen-image-2.1-UC-Q4_0.gguf
else:
    print('✅ [1/3] Diffusion Model 이미 존재함.')

# unet 폴더에도 안전하게 심볼릭 링크 연결
!ln -sf /content/ComfyUI/models/diffusion_models/qwen-image-2.1-UC-Q4_0.gguf /content/ComfyUI/models/unet/ 2>/dev/null || true

# 2. Text Encoder (INT8 ConvRot)
clip_path = '/content/ComfyUI/models/text_encoders/qwen3vl_8b_int8_convrot.safetensors'
if not os.path.exists(clip_path) or os.path.getsize(clip_path) < 1000000:
    print('📥 [2/3] Text Encoder: qwen3vl_8b_int8_convrot.safetensors 다운로드 중...')
    !aria2c --console-log-level=error -c -x 8 -s 8 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors' \
      -d /content/ComfyUI/models/text_encoders -o qwen3vl_8b_int8_convrot.safetensors
else:
    print('✅ [2/3] Text Encoder 이미 존재함.')

# 3. VAE (BF16)
vae_path = '/content/ComfyUI/models/vae/qwen_image_2.1_vae_bf16.safetensors'
if not os.path.exists(vae_path) or os.path.getsize(vae_path) < 1000000:
    print('📥 [3/3] VAE: qwen_image_2.1_vae_bf16.safetensors 다운로드 중...')
    !aria2c --console-log-level=error -c -x 8 -s 8 -k 1M $ua \
      'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors' \
      -d /content/ComfyUI/models/vae -o qwen_image_2.1_vae_bf16.safetensors
else:
    print('✅ [3/3] VAE 이미 존재함.')

print('🎉 모든 모델 파일 확인 완료!')

### 4단계: 원클릭 심플 웹 인터페이스 실행
아래 셀을 실행하면 **Colab 화면 바로 아래에 프롬프트 입력창**이 뜨며, 브라우저 전체 창으로 열 수 있는 `gradio.live` 공개 링크도 제공됩니다!

In [ ]:
import subprocess
import time
import urllib.request
import urllib.parse
import json
import random
import io
import os
import gradio as gr
from PIL import Image

%cd /content/ComfyUI

# 1. 기존 프로세스 종료 (포트 8188 충돌 방지)
!fuser -k 8188/tcp > /dev/null 2>&1 || true
!pkill -9 -f "main.py" > /dev/null 2>&1 || true
time.sleep(2)

# 2. 백그라운드 엔진 실행 (로그 기록)
print('⏳ 백그라운드 이미지 생성 엔진 시작 중...')
log_f = open('/content/comfyui.log', 'w')
engine_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--lowvram', '--preview-method', 'none'],
    stdout=log_f,
    stderr=log_f
)

# 엔진 준비 완료 대기 (최대 60초)
ready = False
for i in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as resp:
            if resp.status == 200:
                print('✅ 백엔드 엔진 준비 완료!')
                ready = True
                break
    except Exception:
        time.sleep(2)

if not ready:
    print('⚠️ 엔진 시작 중 경고: 로그 확인:')
    if os.path.exists('/content/comfyui.log'):
        with open('/content/comfyui.log') as f:
            print(f.read()[-1000:])

# 3. 이미지 생성 함수 정의
def generate_image(prompt, negative_prompt, steps, cfg, width, height, seed, progress=gr.Progress(track_tqdm=True)):
    if not prompt or prompt.strip() == '':
        raise gr.Error('프롬프트를 입력해 주세요!')
    
    if seed == -1 or seed is None:
        seed = random.randint(1, 1000000000)
    
    progress(0.1, desc='작업 요청 생성 중...')
    
    # Qwen-Image-2.1 전용 워크플로우 (type: qwen_image 지정)
    workflow = {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'qwen-image-2.1-UC-Q4_0.gguf'}},
        '2': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'qwen3vl_8b_int8_convrot.safetensors', 'type': 'qwen_image'}},
        '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'qwen_image_2.1_vae_bf16.safetensors'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['2', 0], 'text': prompt}},
        '5': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['2', 0], 'text': negative_prompt}},
        '6': {'class_type': 'EmptyLatentImage', 'inputs': {'width': int(width), 'height': int(height), 'batch_size': 1}},
        '7': {'class_type': 'KSampler', 'inputs': {
            'model': ['1', 0],
            'positive': ['4', 0],
            'negative': ['5', 0],
            'latent_image': ['6', 0],
            'seed': int(seed),
            'steps': int(steps),
            'cfg': float(cfg),
            'sampler_name': 'euler',
            'scheduler': 'normal',
            'denoise': 1.0
        }},
        '8': {'class_type': 'VAEDecode', 'inputs': {'samples': ['7', 0], 'vae': ['3', 0]}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['8', 0], 'filename_prefix': 'Qwen_WebUI'}}
    }

    try:
        data = json.dumps({'prompt': workflow}).encode('utf-8')
        req = urllib.request.Request('http://127.0.0.1:8188/prompt', data=data, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=10) as resp:
            prompt_id = json.loads(resp.read().decode('utf-8'))['prompt_id']
    except Exception as e:
        err_log = ''
        if os.path.exists('/content/comfyui.log'):
            with open('/content/comfyui.log') as f:
                err_log = f.read()[-600:]
        raise gr.Error(f'요청 실패: {e}\n{err_log}')

    progress(0.3, desc='AI 모델이 이미지를 그리는 중 (약 30~50초 소요)...')
    
    filename, subfolder, type_ = None, None, None
    start_time = time.time()
    while time.time() - start_time < 300:
        time.sleep(1)
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:8188/history/{prompt_id}', timeout=5) as resp:
                history = json.loads(resp.read().decode('utf-8'))
                if prompt_id in history:
                    outputs = history[prompt_id].get('outputs', {})
                    if '9' in outputs and 'images' in outputs['9']:
                        img_info = outputs['9']['images'][0]
                        filename = img_info['filename']
                        subfolder = img_info['subfolder']
                        type_ = img_info['type']
                        break
                    status = history[prompt_id].get('status', {})
                    if status.get('status_str') == 'error':
                        raise gr.Error(f'생성 실패: {status}')
        except Exception:
            pass

    if not filename:
        raise gr.Error('이미지 생성 시간 초과 또는 오류가 발생했습니다.')

    progress(0.95, desc='완성된 이미지 변환 중...')
    params = urllib.parse.urlencode({'filename': filename, 'subfolder': subfolder, 'type': type_})
    with urllib.request.urlopen(f'http://127.0.0.1:8188/view?{params}', timeout=10) as resp:
        return Image.open(io.BytesIO(resp.read()))

# 4. 깔끔한 Gradio 웹 UI 생성
with gr.Blocks(theme=gr.themes.Soft(), title='Qwen-Image-2.1 Uncensored') as demo:
    gr.Markdown('# 🎨 Qwen-Image-2.1 Uncensored 원클릭 이미지 생성기')
    gr.Markdown('복잡한 노드 설정 없이, 프롬프트를 입력하고 **[이미지 생성하기]** 버튼만 누르시면 됩니다!')
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_box = gr.Textbox(
                label='📝 프롬프트 (영어로 입력 추천)',
                placeholder='예: a beautiful cybernetic girl with glowing eyes in futuristic seoul, highly detailed, 8k masterpiece',
                lines=4,
                value='a beautiful anime girl with long silver hair in a cherry blossom garden, sunny day, highly detailed, 8k masterpiece'
            )
            neg_prompt_box = gr.Textbox(
                label='🚫 부정 프롬프트 (제외할 특징)',
                placeholder='제외할 요소',
                lines=2,
                value='low quality, blurry, distorted, deformed, bad anatomy, worst quality'
            )
            
            with gr.Accordion('⚙️ 상세 옵션 (기본값 추천)', open=False):
                with gr.Row():
                    width_slider = gr.Slider(512, 1280, value=1024, step=64, label='가로 해상도')
                    height_slider = gr.Slider(512, 1280, value=1024, step=64, label='세로 해상도')
                steps_slider = gr.Slider(15, 35, value=25, step=1, label='생성 스텝수 (Steps, 권장 20~25)')
                cfg_slider = gr.Slider(1.0, 10.0, value=4.0, step=0.5, label='CFG (프롬프트 반영도, 권장 3.5~4.5)')
                seed_input = gr.Number(value=-1, label='시드 번호 (-1이면 랜덤 생성)')
            
            generate_btn = gr.Button('🚀 이미지 생성하기', variant='primary', size='lg')
            
        with gr.Column(scale=1):
            output_img = gr.Image(label='🖼️ 생성된 이미지', type='pil', interactive=False)
            
    generate_btn.click(
        fn=generate_image,
        inputs=[prompt_box, neg_prompt_box, steps_slider, cfg_slider, width_slider, height_slider, seed_input],
        outputs=output_img
    )

# Colab 내부 렌더링 + 외부 접속용 gradio.live 공개 링크 동시 발급
demo.queue().launch(share=True, debug=False)
